# EDA additions for Assignment 4

Most of the EDA is in Assignment 3 and we don't repeat it here. This notebook adds the two things that are
new for Transformers: how long the texts are in XLM-RoBERTa tokens, and how well our CV score followed
the leaderboard in Assignment 3.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/koalla05/deep-learning-assignment4.git"

if "google.colab" in sys.modules and not Path("data").exists():
    subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir("deep-learning-assignment4")

In [2]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base")
cv = pd.read_parquet("data/cv_rows.parquet")
test = pd.read_csv("data/test.csv")

## Text length in tokens

XLM-RoBERTa takes at most 512 tokens, special tokens included. A token is a piece of a word from the
model's vocabulary, so a Ukrainian word is often split into two tokens or more.

In [3]:
def length_row(name, texts):
    lengths = np.array([len(ids) for ids in tokenizer(list(texts))["input_ids"]])
    return {
        "texts": name,
        "count": len(lengths),
        "median": int(np.median(lengths)),
        "95th percentile": int(np.percentile(lengths, 95)),
        "max": int(lengths.max()),
        "over 512": int((lengths > 512).sum()),
    }


pd.DataFrame([
    length_row("training sentences, uk", cv[cv.source == "uk"].text),
    length_row("training sentences, ru", cv[cv.source == "ru"].text),
    length_row("test posts", test.text),
])

,texts,count,median,95th percentile,max,over 512
0,"training sentences, uk",31432,27,54,85,0
1,"training sentences, ru",31956,30,59,217,0
2,test posts,477,73,314,926,8


Test posts are about 2.5 times longer than training sentences, and 8 of them don't fit into 512 tokens.
For training, 128 tokens cover almost every sentence. At prediction time the long posts have to be split
into chunks, otherwise locations near the end get lost.

## CV against the leaderboard in Assignment 3

These are all Assignment 3 submissions that have both a CV score and a public leaderboard score. The two
pilots at the bottom are much weaker than the rest, so we compute the rank correlation with and without
them.

In [4]:
a3 = pd.DataFrame(
    [
        ("word + char BiGRU-CRF, average of 3", 0.896, 0.59475),
        ("word + char BiGRU-CRF, vote 2 of 3", 0.896, 0.59467),
        ("word BiGRU-CRF, vote 2 of 3", 0.891, 0.58914),
        ("word BiGRU-CRF, average of 3", 0.891, 0.58787),
        ("char + 300k models, average of 6", 0.904, 0.58753),
        ("word BiGRU-CRF, fold 2", 0.894, 0.58594),
        ("word BiGRU-CRF, fold 1", 0.896, 0.58308),
        ("word BiGRU-CRF, fold 0", 0.885, 0.57098),
        ("word BiGRU-CRF 300k, vote 2 of 3", 0.899, 0.58170),
        ("GRU + LSTM, average of 6", 0.904, 0.57698),
        ("word BiGRU-CRF 300k, average of 3", 0.899, 0.57313),
        ("BiLSTM-CRF, average of 3", 0.902, 0.56304),
        ("word BiLSTM pilot, no CRF", 0.646, 0.41086),
        ("word BiGRU pilot, no CRF", 0.628, 0.40449),
    ],
    columns=["submission", "cv", "leaderboard"],
)
crf = a3[~a3.submission.str.contains("pilot")]
for name, rows in [("all", a3), ("CRF models only", crf)]:
    rho, p = spearmanr(rows.cv, rows.leaderboard)
    print(f"{name}: n = {len(rows)}, Spearman rho = {rho:.2f}, p = {p:.3f}")
a3

all: n = 14, Spearman rho = 0.20, p = 0.494
CRF models only: n = 12, Spearman rho = -0.28, p = 0.379


,submission,cv,leaderboard
0,"word + char BiGRU-CRF, average of 3",0.896,0.59475
1,"word + char BiGRU-CRF, vote 2 of 3",0.896,0.59467
2,"word BiGRU-CRF, vote 2 of 3",0.891,0.58914
3,"word BiGRU-CRF, average of 3",0.891,0.58787
4,"char + 300k models, average of 6",0.904,0.58753
5,"word BiGRU-CRF, fold 2",0.894,0.58594
6,"word BiGRU-CRF, fold 1",0.896,0.58308
7,"word BiGRU-CRF, fold 0",0.885,0.57098
8,"word BiGRU-CRF 300k, vote 2 of 3",0.899,0.58170
9,"GRU + LSTM, average of 6",0.904,0.57698


CV didn't predict the leaderboard order. Among the 12 CRF models the correlation is even slightly
negative, and neither number is significant. So a better CV score alone isn't a reason to submit a model.